## 1.基本使用

In [13]:
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv
from rich import print as rprint
from pydantic import BaseModel, Field
from typing import Optional

# 加载配置文件
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 获取大模型
model = init_chat_model(
    model_provider="deepseek",
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    
    # 传递非的额外参数
    extra_body = {
        "thinking": {
            "type": "disabled"
        }
    }
)



In [3]:
class Person(BaseModel):
    name: str = Field(description="姓名")
    age: Optional[int] = Field(description="年龄", default=88)
    occupation: str = Field(description="职业")

structured_model = model.with_structured_output(Person)
result = structured_model.invoke("张三是一个软件工程师。提供张三的信息。")
print(result)

name='张三' age=88 occupation='软件工程师'


In [20]:
print(type(result))

<class '__main__.Person'>


In [4]:
from enum import Enum

class Priority(str, Enum):
    """观看方式"""
    THEATER = "影院"
    STREAMING = "流媒体"
    DVD = "DVD"

class MovieModel(BaseModel):
    title: str = Field(description="电影标题")
    director: str = Field(description="导演")
    release_year: int = Field(description="上映年份")
    genre: str = Field(description="类型")
    actors: list[str] = Field(description="主演演员列表")
    score: float = Field(description="评分")
    view: Priority = Field(description="观看方式")

structured_model = model.with_structured_output(MovieModel)
result = structured_model.invoke("请提供电影《盗梦空间》的详细信息。我是在电脑上看的")
print(result)

title='盗梦空间' director='克里斯托弗·诺兰' release_year=2010 genre='科幻/动作/悬疑' actors=['莱昂纳多·迪卡普里奥', '约瑟夫·戈登-莱维特', '艾伦·佩吉', '汤姆·哈迪', '渡边谦', '玛丽昂·歌迪亚', '希里安·墨菲', '迈克尔·凯恩'] score=9.3 view=<Priority.DVD: 'DVD'>


In [16]:
from pydantic import BaseModel, Field
from typing import Literal

class ActorsModel(BaseModel):
    actors: str = Field(description="主演演员名字")
    charactername : str = Field(description="主演演员角色名")
    borndate: str = Field(description="主演演员出生日期")
    age: int = Field(description="主演演员年龄")

class MovieModel(BaseModel):
    title: str = Field(description="电影标题")
    director: str = Field(description="导演")
    release_year: int = Field(description="上映年份")
    genre: str = Field(description="类型")
    actors: list[ActorsModel] = Field(description="主演演员信息")
    score: float = Field(description="评分")
    view: Literal["流媒体", "电影院"] = Field(description="观看方式")

structured_model = model.with_structured_output(MovieModel)
result = structured_model.invoke("请提供电影《盗梦空间》的详细信息，主演信息至少包括三位。我是在电脑上看的。")
print(result)

title='盗梦空间' director='克里斯托弗·诺兰' release_year=2010 genre='科幻/动作/悬疑' actors=[ActorsModel(actors='莱昂纳多·迪卡普里奥', charactername='道姆·柯布', borndate='1974-11-11', age=36), ActorsModel(actors='约瑟夫·高登-莱维特', charactername='亚瑟', borndate='1981-02-17', age=29), ActorsModel(actors='艾伦·佩吉', charactername='阿德里安', borndate='1987-02-21', age=23)] score=9.3 view='流媒体'


## 2.列表提取

In [ ]:
class Person(BaseModel):
    name: str = Field(description="姓名")
    age: Optional[int] = Field(description="年龄", default=88)
    occupation: str = Field(description="职业")

class PersonList(BaseModel):
    """人物列表"""
    people: list[Person] = Field(description="人员列表")    # 多个Person的对象

structured_model = model.with_structured_output(PersonList)
result = structured_model.invoke("张三有30岁，李四有25岁，王五有40岁。")
print(result)


people=[Person(name='张三', age=30, occupation='未知'), Person(name='李四', age=25, occupation='未知'), Person(name='王五', age=40, occupation='未知')]
<class 'pydantic._internal._model_construction.ModelMetaclass'>
